In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, sum as spark_sum, count

spark = SparkSession.builder \
    .appName("TugasMandiri-Pertemuan4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [37]:
print("=== BAGIAN A: Membaca dan Eksplorasi Awal ===")

# Membaca data langsung dari HDFS
hdfs_path="hdfs://localhost:9000/user/areta/tugas4/transaksi_september_2026.csv"

df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

# Menampilkan schema
print("\n1. Schema Data:")
df.printSchema()

# Menampilkan jumlah baris
print(f"\n2. Jumlah Baris: {df.count()}")

# Menampilkan 10 baris pertama
print("\n3. 10 Baris Pertama:")
df.show(10)

=== BAGIAN A: Membaca dan Eksplorasi Awal ===

1. Schema Data:
root
 |-- order_id: string (nullable = true)
 |-- tanggal: date (nullable = true)
 |-- kategori: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- kota: string (nullable = true)


2. Jumlah Baris: 602

3. 10 Baris Pertama:
+--------+----------+--------------------+------------+------------+-----------------+--------+
|order_id|   tanggal|            kategori|unit_terjual|harga_satuan|metode_pembayaran|    kota|
+--------+----------+--------------------+------------+------------+-----------------+--------+
|MAG-2000|2026-08-12|             Fashion|           1|       50000|    Transfer Bank|Magelang|
|MAG-2001|2026-08-18|          Elektronik|           7|       25000|              COD|Magelang|
|MAG-2002|2026-08-07|          Elektronik|           7|       25000|         E-Wallet|Magelang|
|MAG-2003|2026-08-2

In [44]:
from pyspark.sql.functions import avg, col
from pyspark.sql.types import DoubleType

print("\n=== BAGIAN B: Menangani Data Kosong ===")

# 1. Pastikan tipe data harga_satuan bertipe angka (Double)
df_cast = df.withColumn("harga_satuan", col("harga_satuan").cast(DoubleType()))

# 2. Hitung jumlah missing value
jumlah_null = df_cast.filter(col("harga_satuan").isNull()).count()
print(f"Jumlah baris dengan harga_satuan kosong: {jumlah_null}")

# 3. Hitung rata-rata harga_satuan
rata_harga = df_cast.select(avg("harga_satuan")).first()[0]

if rata_harga is not None:
    # Menggunakan round bawaan Python (__builtins__.round) agar tidak bentrok dengan PySpark
    rata_harga_round = __builtins__.round(float(rata_harga), 2)
    df_clean = df_cast.na.fill({"harga_satuan": rata_harga_round})
else:
    df_clean = df_cast

# 4. Cek kembali jumlah missing value setelah ditangani
jumlah_null_after = df_clean.filter(col("harga_satuan").isNull()).count()
print(f"Jumlah missing value setelah ditangani: {jumlah_null_after}")


=== BAGIAN B: Menangani Data Kosong ===
Jumlah baris dengan harga_satuan kosong: 2
Jumlah missing value setelah ditangani: 0


In [46]:
from pyspark.sql.functions import col, sum

# Jawaban Soal 2
df.groupBy("kategori") \
  .agg(sum(col("unit_terjual") * col("harga_satuan")).alias("total_pendapatan")) \
  .orderBy(col("total_pendapatan").desc()) \
  .show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|             Fashion|        80800000|
|          Elektronik|        58700000|
|   Makanan & Minuman|        52450000|
|Kesehatan & Kecan...|        50600000|
|        Rumah Tangga|        40450000|
|            kategori|            NULL|
+--------------------+----------------+



In [47]:
# Jawaban Soal 3
df.groupBy("metode_pembayaran").count().show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD|  160|
|    Transfer Bank|  140|
|     Kartu Kredit|  152|
|         E-Wallet|  148|
|metode_pembayaran|    2|
+-----------------+-----+

